# Modelagem — Score de detração pós-viagem

Este notebook treina e avalia o modelo que estima, para cada jornada encerrada, a probabilidade de o Cliente se tornar Detrator. Ele começa onde o `pre-processamento.ipynb` termina: consome a base analítica já integrada e não repete nenhuma etapa de limpeza ou integração.

Princípios adotados:

- consumir a base analítica pronta, sem reprocessar as fontes brutas;
- importar a lógica de seleção, divisão e transformação de `scripts/preprocessamento_nps.py`, nunca reescrevê-la aqui;
- ajustar imputação, codificação e escala somente no treino;
- manter o conjunto de teste intocado até a etapa de avaliação;
- versionar o notebook sem saídas de execução e sem nenhum dado do parceiro.

O notebook é preenchido por etapas, cada uma com card próprio no board:

| Seção | Conteúdo | Card |
|---|---|---|
| 1 | Contrato de entrada e carga da matriz | #100 |
| 2 | Baselines de referência | #101 |
| 3 | Validação cruzada por Cliente | #102 |
| 4 | Primeiro modelo candidato | #103 |
| 5 | Ajuste de hiperparâmetros | #104 |
| 6 | Métricas de ordenação | #105 |
| 7 | Limiar operacional e matriz de confusão | #106 |
| 8 | Curvas ROC e precisão contra cobertura | #107 |
| 9 | Tabela comparativa | #108 |


## 1. Contrato de entrada

Esta seção fixa o que a etapa de carga e organização dos dados precisa entregar para que a modelagem rode. O contrato existe para que as duas frentes avancem em paralelo: enquanto a organização não estiver pronta, basta que o arquivo abaixo exista com estas colunas para que todo o restante do notebook funcione sem alteração.

**Arquivo esperado:** `data/processed/base_analitica.parquet`, produzido pela seção 13 do `notebooks/pre-processamento.ipynb`. No Colab, o mesmo arquivo pode vir do Drive; a busca de caminho está na célula seguinte e não exige edição de código.

**Colunas obrigatórias.** A base precisa carregar três grupos:

| Grupo | Colunas | Para que serve |
|---|---|---|
| Alvo | `DETRATOR` | variável a ser predita, binária |
| Chaves de partição | `DATA_STD_CONVERTIDA`, `ID_GOLDENRECORD` | corte temporal e agrupamento por Cliente |
| Features | as 11 de `FEATURES_SCORE_POS_VIAGEM` | matriz do modelo |

As 11 features são a allowlist do score pós-viagem definida no módulo de pré-processamento e não são reescritas aqui: `TIER_VIAGEM`, `VOO_TIPO`, `TIPO_ENTRETENIMENTO`, `CANAL_COMPRA`, `SEGMENTO`, `ESTATISTICA_ATRASOSAIDA`, `ATRASO_CHEGADA`, `CANCELAMENTO_VOO`, `ANTECEDENCIA_CANCELAMENTO`, `TEMPO_VOO`, `QTDE_VIAGENS_12M`. Qualquer mudança nessa lista é decisão de modelagem e acontece no módulo, não no notebook.

**Partições.** A divisão não é aleatória. `split.dividir`, chamada dentro de `preparar_matriz` (seção 1.4), aplica corte temporal e por Cliente em três vias — treino, validação e teste —, retirando de cada conjunto anterior todo Cliente que reaparece num conjunto mais recente, porque a base tem dependência intracliente e um Cliente presente em dois lados inflaria o desempenho estimado. A função devolve as três partições como DataFrames completos, e é pelo índice de cada uma que features, alvo e grupos são referenciados no restante do notebook, nunca por posição.

**O que não pode entrar na matriz do modelo.** Nenhuma coluna iniciada por `NPS_` ou `SUB_`. São respostas coletadas no mesmo instrumento que origina o alvo, e o contrato temporal da seção 4.2.3 da documentação as exclui do modelo.

A proibição vale para a matriz, e não para a base analítica. A base carrega essas colunas por construção: `DETRATOR` é derivado de `NPS_PRINCIPAL` dentro da própria `preparar_base_analitica`, e a persistência em Parquet da seção 13 do pré-processamento não descarta coluna alguma. Exigir uma base sem elas faria o notebook falhar sobre uma entrada correta. Por isso as duas verificações têm papéis distintos, e isso é deliberado:

| Célula | Papel | Interrompe a execução |
|---|---|---|
| 1.3 | diagnóstico da base analítica: confere colunas obrigatórias e features da allowlist, e apenas relata quantas colunas de pesquisa vieram junto | sim para coluna obrigatória ausente, não para coluna de pesquisa presente |
| 1.4 | contrato da matriz do modelo: confere que o conjunto de features é exatamente a allowlist e que nenhuma coluna de pesquisa entrou, na entrada e na saída do pré-processador | sim |


### 1.1. Dependências e módulo de pré-processamento

In [1]:
!pip install -q pandas numpy scikit-learn pyarrow matplotlib

import importlib
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Inclui a pasta do módulo tanto no projeto local quanto no Colab. O break evita
# que duas cópias de preprocessamento_nps.py fiquem no caminho ao mesmo tempo,
# o que tornaria ambíguo qual delas o import resolve.
for pasta_codigo in [Path("scripts"), Path("../scripts"), Path("/content/scripts")]:
    if pasta_codigo.exists():
        sys.path.insert(0, str(pasta_codigo.resolve()))
        break

import preprocessamento_nps
importlib.reload(preprocessamento_nps)

from preprocessamento_nps import (
    FEATURES_SCORE_POS_VIAGEM,
    validar_parquet,
)

SEMENTE = 42
np.random.seed(SEMENTE)

# O contrato da seção 1 ancora em 11 features. Uma mudança silenciosa na
# allowlist do módulo precisa falhar aqui, e não apenas aparecer no print.
QUANTIDADE_FEATURES_ESPERADA = 11
if len(FEATURES_SCORE_POS_VIAGEM) != QUANTIDADE_FEATURES_ESPERADA:
    raise AssertionError(
        f"A allowlist mudou: {len(FEATURES_SCORE_POS_VIAGEM)} features, "
        f"esperadas {QUANTIDADE_FEATURES_ESPERADA}. "
        "Atualize o contrato de entrada da seção 1 antes de seguir."
    )

print("features do score pós-viagem:", len(FEATURES_SCORE_POS_VIAGEM))


/bin/bash: pip: command not found


features do score pós-viagem: 11


### 1.2. Localização e carga da base analítica

A busca cobre o projeto local e o Colab, com e sem Drive montado. Se nenhum caminho existir, a célula falha com a instrução do que fazer, em vez de seguir com uma base vazia.


In [2]:
# No Colab, monte o Drive antes de rodar esta célula:
#   from google.colab import drive; drive.mount('/content/drive')

CAMINHOS_POSSIVEIS = [
    Path("data/processed/base_analitica.parquet"),
    Path("../data/processed/base_analitica.parquet"),
    Path("/content/data/processed/base_analitica.parquet"),
    Path("/content/drive/MyDrive/projeto-azul/data/processed/base_analitica.parquet"),
]

caminho_base = next((caminho for caminho in CAMINHOS_POSSIVEIS if caminho.exists()), None)

if caminho_base is None:
    raise FileNotFoundError(
        "Base analítica não encontrada. Gere-a com a seção 13 de notebooks/pre-processamento.ipynb "
        "ou monte o Drive e ajuste CAMINHOS_POSSIVEIS. Caminhos testados: "
        + ", ".join(str(caminho) for caminho in CAMINHOS_POSSIVEIS)
    )

validar_parquet(caminho_base)
df = pd.read_parquet(caminho_base)

print("base analítica:", caminho_base)
print("linhas:", f"{len(df):,}".replace(",", "."), "| colunas:", df.shape[1])


base analítica: ../data/processed/base_analitica.parquet
linhas: 484.915 | colunas: 51


### 1.3. Diagnóstico da base analítica

A célula confere o que a base precisa ter e interrompe a execução se faltar coluna obrigatória ou feature da allowlist. Falhar aqui é barato; descobrir a ausência depois de treinar não é.

A contagem de colunas `NPS_` e `SUB_` é diagnóstico e não motivo de parada, porque a base analítica carrega essas colunas por construção, como explicado na seção 1. O que não pode acontecer é elas alcançarem a matriz do modelo, e é a seção 1.4 que bloqueia isso.


In [3]:
COLUNAS_OBRIGATORIAS = {"DETRATOR", "DATA_STD_CONVERTIDA", "ID_GOLDENRECORD"}

faltando = COLUNAS_OBRIGATORIAS - set(df.columns)
if faltando:
    raise KeyError(f"A base não atende ao contrato de entrada. Colunas ausentes: {sorted(faltando)}")

features_ausentes = [coluna for coluna in FEATURES_SCORE_POS_VIAGEM if coluna not in df.columns]
if features_ausentes:
    raise KeyError(f"Features da allowlist ausentes na base: {features_ausentes}")

# Diagnóstico, e não trava: a base analítica carrega as respostas da pesquisa
# por construção, já que DETRATOR é derivado de NPS_PRINCIPAL. O bloqueio
# efetivo acontece sobre a matriz do modelo, na seção 1.4.
colunas_pesquisa = [coluna for coluna in df.columns if coluna.startswith(("NPS_", "SUB_"))]
print("colunas de pesquisa na base analítica:", len(colunas_pesquisa), "(diagnóstico, não bloqueia)")
print("prevalência de Detrator:", f"{df['DETRATOR'].mean():.2%}")


colunas de pesquisa na base analítica: 24 (diagnóstico, não bloqueia)
prevalência de Detrator: 20.44%


### 1.4. Matriz de modelagem e contrato da matriz

`preparar_matriz`, de `src/matriz.py`, executa numa chamada as três etapas que precisam
acontecer nesta ordem: particionar por data e por Cliente, selecionar a allowlist de features e
ajustar o pré-processador **apenas no treino**. Inverter as duas últimas seria inofensivo;
ajustar o pré-processador antes de particionar não, porque a mediana do treino passaria a
carregar informação de 2026 e a métrica de teste mediria um modelo que já viu o período que
deveria prever.

As datas de corte não são escolhidas aqui. Elas vêm do registro de decisão em
`documents/extras/politica-de-particionamento-temporal.md`, que é a fonte única: repeti-las como
constante do notebook criaria uma segunda verdade que poderia divergir da documentada.

`sem_data="treino"` envia ao treino qualquer linha sem data — hoje cerca de 120 mil na base
analítica, remanescentes de respostas antigas que a Azul não datou. Enviá-las ao treino em vez
de descartá-las evita perder um quinto da base; a garantia de que isso não corrompe o corte
temporal vem de `verificar_anterioridade_sem_data`, que prova pela ordem de registro
(`RESPONDENT_ID`) que essas linhas antecedem o período datado. `split.dividir` chama essa
verificação por conta própria sempre que existir alguma linha sem data, interrompendo a
execução se a prova deixar de valer.

O contrato da matriz é cobrado nas duas pontas do pré-processador. Na entrada, pega uma allowlist
alterada; na saída, uma transformação que reintroduza uma coluna de pesquisa sob outro nome.
Qualquer uma das duas interrompe a execução.


In [4]:
for pasta_codigo in [Path("src"), Path("../src"), Path("/content/src")]:
    if pasta_codigo.exists():
        sys.path.insert(0, str(pasta_codigo.resolve()))
        break

import matriz
importlib.reload(matriz)
from matriz import preparar_matriz, resumo_da_matriz

# Datas do registro de decisão do #127, não escolhidas aqui.
CORTE_VALIDACAO = "2025-07-01"
CORTE_TESTE = "2026-01-01"

preparo = preparar_matriz(
    df,
    corte_validacao=CORTE_VALIDACAO,
    corte_teste=CORTE_TESTE,
    sem_data="treino",
)

x_treino, x_validacao, x_teste = (preparo["x"][k] for k in ("treino", "validacao", "teste"))
y_treino, y_validacao, y_teste = (preparo["y"][k] for k in ("treino", "validacao", "teste"))
grupos_treino = preparo["grupos"]["treino"]
preprocessador = preparo["preprocessador"]
x_treino_transformado = preparo["matrizes"]["treino"]
x_validacao_transformado = preparo["matrizes"]["validacao"]
x_teste_transformado = preparo["matrizes"]["teste"]
numericas = preparo["metadados"]["numericas"]
categoricas = preparo["metadados"]["categoricas"]
metadados_divisao = preparo["metadados"]

display(resumo_da_matriz(preparo))

anterioridade = metadados_divisao["anterioridade_sem_data"]
# verificar_anterioridade_sem_data devolve dois formatos: um quando ha linha
# sem data para verificar, outro (sem as chaves de correlacao) quando o bloco
# esta vazio, que e o caso normal na base atual.
if anterioridade["linhas_sem_data"]:
    print("linhas sem data enviadas ao treino:", f"{anterioridade['linhas_sem_data']:,}".replace(",", "."))
    print("  correlação de Spearman entre ordem de registro e data:", anterioridade["correlacao_spearman"])
    print("  maior id sem data:", anterioridade["maior_id_sem_data"],
          "< menor id datado:", anterioridade["menor_id_datado"])
else:
    print("nenhuma linha sem data na base: a verificação de anterioridade não se aplica")
print("removidas por Cliente recorrente:",
      f"{metadados_divisao['linhas_removidas_por_recorrencia']:,}".replace(",", "."))
print("features:", len(metadados_divisao["features_usadas"]),
      "| numéricas:", len(numericas), "| categóricas:", len(categoricas),
      "| colunas da matriz:", metadados_divisao["colunas_da_matriz"])


,n,clientes,prevalencia_pct,colunas_matriz
particao,,,,
treino,341962,307510,20.43,31
validacao,48301,47560,21.60,31
teste,53486,52069,20.41,31


linhas sem data enviadas ao treino: 120.000
  correlação de Spearman entre ordem de registro e data: 0.9999
  maior id sem data: 31673808 < menor id datado: 31673817
removidas por Cliente recorrente: 41.063
features: 11 | numéricas: 5 | categóricas: 6 | colunas da matriz: 31


### 1.5. Composição das partições (CR05 do #127/#128)

Esta seção não reparticiona: `split.resumo` recebe `preparo["particoes"]` e
`preparo["metadados"]`, já calculados na 1.4, e devolve n, intervalo de datas,
Clientes e percentual do total por partição. Chamar `split.dividir` de novo aqui
geraria a mesma divisão duas vezes no mesmo notebook, com o risco de as duas
chamadas divergirem se uma delas for editada e a outra não.


In [5]:
from split import resumo

tabela_particoes = resumo(preparo["particoes"], metadados=preparo["metadados"])
print(tabela_particoes)


                n  pct_do_total  clientes data_inicio    data_fim  sem_data  \
particao                                                                      
treino     341962          70.5    307510  2024-01-06  2025-06-30    111217   
validacao   48301          10.0     47560  2025-07-01  2025-12-31         0   
teste       53486          11.0     52069  2026-01-01  2026-06-30         0   

           taxa_alvo_pct  
particao                  
treino             20.43  
validacao          21.60  
teste              20.41  


## 2. Baselines de referência

Um número de desempenho só significa alguma coisa quando existe um piso para compará-lo. Esta seção estabelece dois pisos, e todo modelo apresentado depois é julgado contra eles.

O primeiro é a **regra da classe majoritária**, que não aprende nada e existe para expor o quanto a acurácia engana numa base em que cerca de um em cada cinco Clientes é Detrator. O segundo é uma **regressão logística**, modelo linear e interpretável, que mede quanto do sinal já é capturado sem recorrer a interação entre variáveis. É contra esse segundo piso que a escolha de um método baseado em árvores, na seção 4, precisa se justificar.

Os dois são ajustados apenas na partição de treino e medidos sobre a mesma partição de teste que o modelo candidato usará, sem o que a comparação não seria válida.

**Escopo desta seção.** A acurácia e a precisão média aparecem aqui como piso de comparação, não como a resposta ao item (c) da Seção 4.3. A escolha e a explicação das três métricas oficiais do modelo são daquele item e ficam na seção 6 deste notebook.

### 2.1. Prevalência e função de avaliação

A prevalência da classe Detrator é o número que dá sentido a todo o resto: é ela que a precisão média de um modelo sem informação reproduz, e é dela que a acurácia do baseline trivial se aproxima por complemento.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, average_precision_score,
                             roc_auc_score)

print(f"prevalência de Detrator no treino: {y_treino.mean():.2%}")
print(f"prevalência de Detrator no teste:  {y_teste.mean():.2%}")


def avaliar_no_teste(nome, modelo):
    """Mede um modelo já treinado sobre a partição de teste temporal.

    A acurácia entra ao lado da precisão média de propósito. É a distância
    entre as duas que evidencia o efeito do desbalanceamento, e é isso que
    justifica não apresentar acurácia como medida de qualidade neste problema.

    O ROC-AUC entra porque a tabela comparativa da seção 9 cobra as três
    métricas oficiais definidas na seção 4.1.3, e ele é a única das três que
    já pode ser medida aqui: depende apenas da ordenação do score. Revocação
    em Detrator e precisão dependem do ponto de corte, que só é decidido na
    seção 7, e medi-las agora seria fixar um limiar arbitrário antes da
    seção que existe para escolhê-lo.
    """
    probabilidades = modelo.predict_proba(x_teste_transformado)[:, 1]
    return {
        "modelo": nome,
        "acuracia": accuracy_score(y_teste, modelo.predict(x_teste_transformado)),
        "precisao_media": average_precision_score(y_teste, probabilidades),
        "roc_auc": roc_auc_score(y_teste, probabilidades),
    }

### 2.2. Baseline trivial: regra da classe majoritária

O classificador responde sempre a classe mais frequente do treino, que é a de não Detrator. Ele não olha nenhuma feature.

O resultado esperado é uma acurácia próxima de quatro quintos, acompanhada de uma precisão média próxima da prevalência. É essa contradição que interessa: um modelo que erra todos os Detratores da base ainda assim exibe acurácia alta. A probabilidade que ele emite é degenerada, constante para toda linha, e é por isso que a precisão média desaba para o piso. Pela mesma razão o ROC-AUC fica em 0,5: sem variação no score não existe ordenação, e 0,5 é exatamente o valor de quem ordena ao acaso.

In [ ]:
trivial = DummyClassifier(strategy="most_frequent", random_state=SEMENTE)
trivial.fit(x_treino_transformado, y_treino)

resultado_trivial = avaliar_no_teste("classe majoritária", trivial)
print(f"acurácia:       {resultado_trivial['acuracia']:.4f}")
print(f"precisão média: {resultado_trivial['precisao_media']:.4f}")
print(f"ROC-AUC:        {resultado_trivial['roc_auc']:.4f}")

### 2.3. Baseline linear: regressão logística

O segundo piso já aprende, mas apenas efeitos aditivos: cada variável contribui com um peso fixo, sem que o modelo represente combinações entre elas.

O parâmetro de balanceamento de classe é declarado explicitamente. Sem ele a logística se ajusta à classe majoritária e passa a quase não prever Detrator, o que a tornaria indistinguível do baseline trivial e inútil como piso. A matriz chega aqui já escalonada pelo pré-processador da seção 1.4, condição para que os coeficientes sejam comparáveis entre variáveis de unidades diferentes.

In [ ]:
from sklearn.linear_model import LogisticRegression

logistica = LogisticRegression(
    class_weight="balanced",   # declarado: sem isso o modelo colapsa na classe majoritária
    max_iter=5000,          # lbfgs converge em 1010 iterações nesta matriz; 1000 para antes
    random_state=SEMENTE,
)
logistica.fit(x_treino_transformado, y_treino)

resultado_logistica = avaliar_no_teste("regressão logística", logistica)
print(f"acurácia:       {resultado_logistica['acuracia']:.4f}")
print(f"precisão média: {resultado_logistica['precisao_media']:.4f}")
print(f"ROC-AUC:        {resultado_logistica['roc_auc']:.4f}")

### 2.4. Coeficientes da regressão logística

Os coeficientes ficam acessíveis no notebook para leitura posterior. Eles indicam a direção com que cada variável empurra o risco de detração, e voltam a ser usados na discussão de resultados, quando for preciso dizer se o modelo candidato encontrou algo que o piso linear não via.

In [ ]:
coeficientes_logistica = pd.DataFrame({
    "feature": preprocessador.get_feature_names_out(),
    "coeficiente": logistica.coef_[0],
}).sort_values("coeficiente", key=abs, ascending=False, ignore_index=True)

print("intercepto:", float(logistica.intercept_[0]))
coeficientes_logistica.head(15)

### 2.5. Resultado consolidado dos dois pisos

A tabela abaixo é o piso de comparação que as seções seguintes consomem. A leitura que importa: o baseline trivial exibe a maior acurácia da tabela e a menor precisão média, e a regressão logística inverte essa relação. O ROC-AUC completa a leitura: o trivial fica no 0,5 de quem não ordena nada, e é a distância da logística para esse 0,5 que mede o sinal já capturado por um modelo puramente aditivo. Qualquer modelo apresentado depois precisa superar a segunda linha na precisão média para que seu ganho seja atribuível ao modelo, e não ao desbalanceamento da base.

In [ ]:
tabela_baselines = pd.DataFrame([resultado_trivial, resultado_logistica])

ganho = resultado_logistica["precisao_media"] - resultado_trivial["precisao_media"]
print(f"ganho da logística sobre o trivial na precisão média: {ganho:+.4f}")
if ganho <= 0:
    print("ATENÇÃO: a logística não superou o piso trivial; revise o balanceamento antes de seguir.")

tabela_baselines

## 3. Validação cruzada por Cliente

`GroupKFold` de 5 folds dentro do treino, agrupando por `ID_GOLDENRECORD` para que nenhum Cliente apareça em dois folds.

*A preencher no card #102.*


## 4. Primeiro modelo candidato

Classificador de gradient boosting sobre árvores, escolha justificada pela ausência de preditor dominante e pela interação entre fidelização e falha operacional documentadas na seção 4.2.1.

*A preencher no card #103.*


## 5. Ajuste de hiperparâmetros

Busca sobre taxa de aprendizado, profundidade e número de iterações, avaliada apenas nos folds de validação.

*A preencher no card #104.*


## 6. Métricas de ordenação do score

Precisão média, área sob a curva ROC e escore de Brier, calculadas sobre o conjunto de teste.

*A preencher no card #105.*


## 7. Limiar operacional e matriz de confusão

Ponto de corte derivado da capacidade diária de contato da equipe de Experiência do Cliente, e não fixado em 0,5 por omissão.

*A preencher no card #106.*


## 8. Curvas ROC e de precisão contra cobertura

Figuras salvas em `assets/`, no mesmo padrão das da seção 4.2.

*A preencher no card #107.*


## 9. Tabela comparativa

Baselines e modelo candidato nas mesmas métricas e sobre a mesma partição de teste.

*A preencher no card #108.*
